In [1]:
import duckdb

con = duckdb.connect(database=":memory:")

parquet_file_path = "cleaned.parquet"

# create a vew 
con.sql(f"create or replace view silver_mining as select * from read_parquet('{parquet_file_path}')")

# Aggregation & Column Filtering
# Query 1 - What are the top 5 highest energy expenses in British Columbia for the most recent
query_top_expenses = """
    select 
        ref_date,
        energy_type,
        value as expensive_in_thousands
    from silver_mining
    where geography = 'British Columbia'
    and energy_type like '%expenses%'
    order by value desc
    limit 20;
"""
# con.sql(query_top_expenses).show()

# Query 2: Multi-Dimensional Aggregation (Pivot-like analysis)
# Total energy expenses broken down by Province and Industry.

query_provincial_summary = """
    select 
        geography,
        naics_industry,
        energy_type,
        sum(value) as total_expense_thousands,
        round(avg(value), 2) as avg_expense_thousands,
    from silver_mining
    where contains(energy_type, 'expenses')
    and geography not in ('Canada', 'Atlantic Region', 'Prairie Region', 'Northern Territories', 'Northwest Territories')
    group by
        geography,
        naics_industry,
        energy_type
    limit 200;;
"""
# con.sql(query_provincial_summary).show()

# Query 3: Window Function (Data Engineering Gold Layer pattern)
# Rank the energy types by expense within each province for the latest year (use 2 since the data is old).
query_ranked_expenses_gemini = """
    SELECT 
        ref_date,
        geography,
        energy_type,
        value,
        DENSE_RANK() OVER (
            PARTITION BY geography 
            ORDER BY value DESC
        ) as expense_rank
    FROM silver_mining
    WHERE energy_type LIKE '%expenses%'
      AND ref_date = (SELECT MAX(ref_date) FROM silver_mining)
    QUALIFY expense_rank <= 3 -- Top 3 per province
    ORDER BY geography, expense_rank;
"""
# con.sql(query_ranked_expenses_gemini).show()

query_ranked_expenses = """
    select
        ref_date,
        geography,
        energy_type,
        value,
        dense_rank() over (partition by geography order by value desc) as rnk
    from silver_mining
    where ref_date = extract(year from CURRENT_DATE) - 2
    and geography not in ('Canada', 'Atlantic Region', 'Prairie Region', 'Northern Territories', 'Northwest Territories')
    and value is not null and value != 0
    and energy_type like '%expenses%'
    qualify rnk <=3
    order by geography, rnk
    limit 200;
"""
con.sql(query_ranked_expenses).show()


┌──────────┬──────────────────┬───────────────────────────────────────────────┬──────────┬───────┐
│ ref_date │    geography     │                  energy_type                  │  value   │  rnk  │
│  int32   │     varchar      │                    varchar                    │  double  │ int64 │
├──────────┼──────────────────┼───────────────────────────────────────────────┼──────────┼───────┤
│     2024 │ Alberta          │ Diesel fuel, expenses (thousands of dollars)  │  46874.0 │     1 │
│     2024 │ Alberta          │ Electricity, expenses (thousands of dollars)  │    894.0 │     2 │
│     2024 │ Alberta          │ Diesel fuel, expenses (thousands of dollars)  │    619.0 │     3 │
│     2024 │ British Columbia │ Total, energy expenses (thousands of dollars) │  70540.0 │     1 │
│     2024 │ British Columbia │ Diesel fuel, expenses (thousands of dollars)  │  49372.0 │     2 │
│     2024 │ British Columbia │ Electricity, expenses (thousands of dollars)  │  10795.0 │     3 │
│     2024